# Pertemuan 14
## Transportation in Supply Chains

Optimasi rute pengiriman ekspor kopi dari pelabuhan-pelabuhan utama Indonesia menuju Pelabuhan Rotterdam, Belanda.

---
## 1. Setup

In [ ]:
# %pip install ortools

   ---------------------------------------- 0.0/24.7 MB ? eta -:--:--
   -- ------------------------------------- 1.8/24.7 MB 14.1 MB/s eta 0:00:02
   ------- -------------------------------- 4.5/24.7 MB 13.3 MB/s eta 0:00:02
   ----------- ---------------------------- 7.1/24.7 MB 13.2 MB/s eta 0:00:02
   ---------------- ----------------------- 10.0/24.7 MB 13.1 MB/s eta 0:00:02
   -------------------- ------------------- 12.6/24.7 MB 13.1 MB/s eta 0:00:01
   ------------------------ --------------- 15.2/24.7 MB 13.1 MB/s eta 0:00:01
   ---------------------------- ----------- 17.8/24.7 MB 13.1 MB/s eta 0:00:01
   --------------------------------- ------ 20.4/24.7 MB 13.0 MB/s eta 0:00:01
   ------------------------------------- -- 23.1/24.7 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 24.7/24.7 MB 12.8 MB/s  0:00:01

   ---------------------------------------- 0/4 [protobuf]
   ---------------------------------------- 0/4 [protobuf]
   ------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 2. Dataset Rute

In [1]:
import pandas as pd

rute = pd.DataFrame({
    'asal':          ['Belawan', 'Tanjung Priok', 'Tanjung Perak', 'Makassar'],
    'tujuan':        ['Rotterdam'] * 4,
    'jarak_km':      [9800,       11200,           12400,           13800],
    'biaya_rp_ton':  [12000000,   13500000,        15000000,        17500000],
    'waktu_jam':     [480,        552,             600,             648],
    'kapasitas_ton': [1000,       1500,            1200,            800],
    'stok_tersedia': [2500,       4000,            3200,            1800],   # ton
})

rute

,asal,tujuan,jarak_km,biaya_rp_ton,waktu_jam,kapasitas_ton,stok_tersedia
0,Belawan,Rotterdam,9800,12000000,480,1000,2500
1,Tanjung Priok,Rotterdam,11200,13500000,552,1500,4000
2,Tanjung Perak,Rotterdam,12400,15000000,600,1200,3200
3,Makassar,Rotterdam,13800,17500000,648,800,1800


---
## 3. Optimasi Rute dengan OR-Tools

Target: penuhi kebutuhan ekspor ke Rotterdam sebesar 5.000 ton dengan biaya minimum dan waktu tempuh ≤ 30 hari (720 jam).

In [2]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver('SCIP')

KEBUTUHAN = 5000  # ton
BATAS_WAKTU = 720  # jam (30 hari)

# Variabel: jumlah ton yang dikirim dari setiap pelabuhan asal
x = [
    solver.NumVar(0.0, float(rute.loc[i, 'stok_tersedia']), f"x_{rute.loc[i, 'asal']}")
    for i in range(len(rute))
]

# Constraint 1: total pengiriman >= kebutuhan
solver.Add(sum(x) >= KEBUTUHAN)

# Constraint 2: waktu pengiriman <= batas waktu
for i in range(len(rute)):
    solver.Add(x[i] * rute.loc[i, 'waktu_jam'] / rute.loc[i, 'kapasitas_ton'] <= BATAS_WAKTU)

# Objective: minimasi biaya
solver.Minimize(sum(x[i] * rute.loc[i, 'biaya_rp_ton'] for i in range(len(rute))))

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Solusi optimal ditemukan!")
    print(f"Total biaya: Rp {solver.Objective().Value():,.0f}\n")
    hasil = []
    for i in range(len(rute)):
        if x[i].solution_value() > 0:
            hasil.append({
                'asal': rute.loc[i, 'asal'],
                'jumlah_ton': round(x[i].solution_value(), 1),
                'biaya_rp': round(x[i].solution_value() * rute.loc[i, 'biaya_rp_ton']),
                'waktu_jam': rute.loc[i, 'waktu_jam'],
            })
    print(pd.DataFrame(hasil))
else:
    print("Solusi tidak ditemukan.")

Solusi optimal ditemukan!
Total biaya: Rp 67,823,913,043

            asal  jumlah_ton     biaya_rp  waktu_jam
0        Belawan      1500.0  18000000000        480
1  Tanjung Priok      1956.5  26413043478        552
2  Tanjung Perak      1440.0  21600000000        600
3       Makassar       103.5   1810869565        648


---
## 4. AI Agent: Rekomendasi Rute (Groq)

In [ ]:
from groq import Groq
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

data_str = rute.to_string(index=False)

prompt = f"""
Data rute pengiriman ekspor kopi dari pelabuhan Indonesia ke Rotterdam:
{data_str}

Kebutuhan Rotterdam: 5000 ton
Batas waktu pengiriman: 30 hari

Cari rute distribusi paling murah dengan batas waktu pengiriman 30 hari.
Berikan:
1. Rute yang dipilih dan alasannya
2. Total biaya estimasi
3. Rekomendasi untuk efisiensi jangka panjang
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Berikut adalah jawaban untuk pertanyaan Anda:

**1. Rute yang dipilih dan alasannya:**

Untuk memenuhi kebutuhan 5.000 ton dengan batas waktu pengiriman 30 hari (720 jam), kita perlu mengombinasikan beberapa pelabuhan asal karena tidak ada satu pelabuhan pun yang sanggup memenuhi seluruh kebutuhan sendirian.

Berdasarkan data, Pelabuhan Belawan memiliki biaya termurah per ton (Rp 12.000.000) dan jarak terdekat ke Rotterdam (9.800 km), sehingga menjadi prioritas utama. Selanjutnya, Tanjung Priok dipilih karena memiliki kapasitas kapal terbesar (1.500 ton) dan biaya yang masih kompetitif (Rp 13.500.000/ton). Tanjung Perak ditambahkan untuk menutupi sisa kebutuhan dengan biaya menengah (Rp 15.000.000/ton), dan sebagian kecil sisanya dipenuhi dari Makassar meski biayanya paling tinggi (Rp 17.500.000/ton).

Dengan demikian, rute yang dipilih adalah:
- Belawan → Rotterdam (±1.500 ton)
- Tanjung Priok → Rotterdam (±1.957 ton)
- Tanjung Perak → Rotterdam (±1.440 ton)
- Makassar → Rotterdam (±1